# 02 · Les baselines (PF.5)

Le livrable de l'étape PF.5 : ce notebook **exécuté** et les premières lignes du journal `experiments/runs.jsonl`. Deux modèles, sous le protocole figé (`docs/protocole.md`, tag `protocole`) : le **prédicteur naïf** (`models.build_naive`), le plancher que tout modèle doit battre, et le **modèle classique** (`models.build_classical`), un `Pipeline` scikit-learn dont le prétraitement est réappris dans chaque fold. La validation croisée (`train.cross_validate`) ne voit que l'entraînement.

Chaque étape tourne dans `with step(...)` : tant qu'une fonction de ton paquet lève `NotImplementedError`, ou qu'une étape précédente n'a pas tourné, la cellule affiche ⏳ au lieu d'une erreur, et « Tout exécuter » va jusqu'au bout. Le notebook n'a pas de code d'apprentissage à lui : il appelle ton paquet (`src/projet/`). Ajoute tes propres cellules (figures, vérifications) là où le disent les `TODO`.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True   # True: configs/fast.yaml (CPU, < 10 min); False: configs/full.yaml (🚀 GPU Colab advised)
REPO_ON_DRIVE = "/content/drive/MyDrive/TODO-your-repository"   # on Colab only: where your repository is cloned

import os
import sys
from contextlib import contextmanager
from pathlib import Path

import pandas as pd

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    os.chdir(REPO_ON_DRIVE)

ROOT = Path.cwd().resolve()
while not (ROOT / "configs" / "fast.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "configs" / "fast.yaml").exists():
    raise FileNotFoundError("Projet introuvable : ouvre ce notebook depuis ton dépôt (celui qui a configs/fast.yaml).")
os.chdir(ROOT)                                    # the paths of the configuration are relative to the project
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))         # your package, even without pip install -e


class Pending(Exception):
    """An earlier step of the notebook did not run."""


def need(**values):
    """Raise Pending if one of the values is still None (the step that computes it is not done)."""
    missing = [name for name, value in values.items() if value is None]
    if missing:
        raise Pending("attends d'abord : " + ", ".join(missing))


@contextmanager
def step(title):
    """Run one step: a function not written yet, or a missing earlier step, prints ⏳ instead of an error."""
    try:
        yield
    except NotImplementedError as err:
        print(f"⏳ {title} : pas encore écrit ({err})")
    except Pending as err:
        print(f"⏳ {title} : {err}")


from projet.config import default_config_path, load_config  # noqa: E402

CONFIG_PATH = default_config_path(FAST_MODE)
config = load_config(CONFIG_PATH)
print(f"Projet : {ROOT.name} · mode {'rapide' if FAST_MODE else 'complet'} ({CONFIG_PATH.name}) · tâche : {config.task}")

## 1. La validation croisée, journalisée

La cellule lance la commande `projet baselines` (`src/projet/cli.py`) : validation croisée des deux modèles, un résumé JSON par modèle, et une ligne par run dans le journal (`train.log_run`), même si un run échoue.

In [ ]:
with step("baselines (projet baselines : validation croisée et journal)"):
    from projet.cli import main as projet

    code = projet(["baselines", "--config", str(CONFIG_PATH)])
    if code == 3:
        raise NotImplementedError("une fonction des baselines (le message ci-dessus la nomme)")
    print("✅ terminé" if code == 0 else f"❌ code de sortie {code} : lis le message ci-dessus")

## 2. Le journal

Les derniers runs du journal, lus par `train.read_runs`. Le tableau de synthèse du rapport ne s'écrit jamais à la main : `python scripts/summarize_runs.py --out experiments/summary.md` (PF.7).

In [ ]:
with step("le journal (train.read_runs)"):
    from projet.train import read_runs

    runs = read_runs(config.runs_file)
    need(runs=runs or None)
    rows = []
    for run in runs[-10:]:
        cv = (run.get("metrics") or {}).get("cv") or {}
        rows.append({"run_id": run["run_id"], "modèle": run.get("model"), "statut": run["status"],
                     config.metric: cv.get("mean"), "écart-type": cv.get("std"), "secondes": run.get("seconds")})
    display(pd.DataFrame(rows))

## Ce que tu retiens

Recopie dans `docs/rapport.md` (« Les baselines ») : les deux scores de validation croisée (moyenne ± écart-type sur les folds), les `run_id`, et ce qu'ils disent du problème (une baseline classique très au-dessus du naïf : il y a du signal ; à peine au-dessus : le deep learning devra le prouver). Puis commite le journal.

**La suite (PF.6)** : le modèle deep learning, entraîné par `projet train --config configs/fast.yaml`, puis `configs/full.yaml` (🚀 GPU Colab conseillé).